# CNN ile Fashion-MNIST Görüntü Sınıflandırma Gradio Uygulaması

Eğitilmiş CNN'in ağırlıkları bu uygulamanın içerisine gömülmüştür.

In [2]:
!pip install -q gradio

## Gömülü Model Ağırlıkları

`fashion_cnn_best.pth` dosyasının (eğitimdeki en iyi epoch) base64 hâli. Bu hücreye dokunmayınız, sadece çalıştırınız

Satır sayısı yüksektir. Hücre çalıştırıldıktan sonra bu hücre başlığının altına gizlenmesi önerilir. ("Gömülü Model Ağırlıkları" başlığının yanındaki çentik ile)

In [3]:
# fashion_cnn_best.pth (base64) + bütünlük kontrolü için SHA-256
MODEL_SHA256 = "35b5a4d264c589e818235ce5a7f57e57bb3ebd2e2e8a5c213b348ff60eb1b379"
MODEL_B64 = """
...
"""

## Gömülü Ağırlıkların Çözülmesi ve Doğrulanması - Model

In [4]:
import io, base64, hashlib, time
import numpy as np
import torch
import torch.nn as nn

# ---------------------------------------------------------------- Gömülü ağırlıkları çöz ve doğrula
raw = base64.b64decode("".join(MODEL_B64.split()))
if hashlib.sha256(raw).hexdigest() != MODEL_SHA256:
    raise RuntimeError("Gömülü model verisi bozuk (SHA-256 uyuşmuyor). Önceki hücreyi eksiksiz çalıştırdığınızdan emin olun.")
print(f"Gömülü model doğrulandı: {len(raw)/1024:.0f} KB")

# ---------------------------------------------------------------- Model (eğitimdekiyle birebir aynı)
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128 * 3 * 3, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
model.load_state_dict(torch.load(io.BytesIO(raw), map_location=device))
model.eval()
print("Model yüklendi | cihaz:", device, "| parametre sayısı:", f"{sum(p.numel() for p in model.parameters()):,}")

# ---------------------------------------------------------------- Sınıflar ve normalizasyon (eğitimdekiyle aynı)
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
CLASS_TR    = ["Tişört", "Pantolon", "Kazak", "Elbise", "Mont",
               "Sandalet", "Gömlek", "Spor ayakkabı", "Çanta", "Bilekli bot"]
MEAN, STD = 0.2860, 0.3530

Gömülü model doğrulandı: 956 KB
Model yüklendi | cihaz: cuda | parametre sayısı: 241,994


## Ön İşleme

Fashion-MNIST görüntüleri **koyu zemin üzerinde açık renkli, ortalanmış, çerçeveyi dolduran 28×28 gri tonlu** nesnelerdir.
Telefon fotoğrafları ise renkli, büyük ve arka planlıdır; bu yüzden fotoğraf önce eğitim verisine benzetilir:

1. EXIF yönünü düzelt, gri tona çevir
2. Arka plan parlaklığını (kenar piksellerinin medyanı) bul; açıksa görüntüyü ters çevir
3. Arka planı siyaha çek, kontrastı aç
4. Nesneyi bulup etrafını kırp
5. En-boy oranını bozmadan kareye tamamla ve 28×28'e küçült

In [5]:
from PIL import Image, ImageOps

BG_MODES = ["Otomatik", "Koyu arka plan (ters çevirme)", "Açık arka plan (ters çevir)"]


def _border_median(a, frac=0.05):
    """Görüntünün kenar piksellerinin medyanı = arka plan parlaklığı."""
    b = max(1, int(min(a.shape) * frac))
    border = np.concatenate([a[:b, :].ravel(), a[-b:, :].ravel(),
                             a[:, :b].ravel(), a[:, -b:].ravel()])
    return float(np.median(border))


def preprocess_to_28(img, bg_mode="Otomatik"):
    """Telefon fotoğrafını Fashion-MNIST formatına (koyu zemin üzerinde açık nesne, 28x28, gri ton) çevirir."""
    # 1) EXIF yönünü düzelt (telefon fotoğrafları), gri tona çevir, büyükse küçült
    img = ImageOps.exif_transpose(img).convert("L")
    img.thumbnail((512, 512))
    a = np.asarray(img, dtype=np.float32)

    # 2) Arka plan rengini bul; açıksa ters çevir
    bg = _border_median(a)
    invert = (bg > 127) if bg_mode == "Otomatik" else bg_mode.startswith("Açık")
    if invert:
        a, bg = 255.0 - a, 255.0 - bg

    # 3) Arka planı siyaha çek, kontrastı aç
    a = np.clip(a - bg, 0, None)
    if a.max() > 0:
        a = a / a.max() * 255.0

    # 4) Nesneyi bul, etrafını kırp (Fashion-MNIST nesneleri çerçeveyi dolduruyor)
    mask = a > 40
    if mask.sum() > 20:
        ys, xs = np.where(mask)
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        m = int(0.08 * max(y1 - y0, x1 - x0))
        y0, y1 = max(0, y0 - m), min(a.shape[0], y1 + m)
        x0, x1 = max(0, x0 - m), min(a.shape[1], x1 + m)
        a = a[y0:y1, x0:x1]

    # 5) En-boy oranını bozmadan kareye tamamla (siyah dolgu), 28x28'e küçült
    h, w = a.shape
    s = max(h, w)
    canvas = np.zeros((s, s), dtype=np.float32)
    top, left = (s - h) // 2, (s - w) // 2
    canvas[top:top + h, left:left + w] = a
    out = Image.fromarray(canvas.astype(np.uint8)).resize((28, 28), Image.LANCZOS)
    return np.asarray(out, dtype=np.uint8)

## Tahmin Fonksiyonu ve Gradio Arayüzü

In [8]:
import gradio as gr
from PIL import Image

DISPLAY_NAMES = [f"{en} ({tr})" for en, tr in zip(CLASS_NAMES, CLASS_TR)]


def predict(image, bg_mode):
    if image is None:
        return None, None, "Lütfen bir fotoğraf yükleyiniz."

    arr28 = preprocess_to_28(image, bg_mode)                      # (28, 28) uint8

    x = torch.from_numpy(arr28).float().div(255.0)
    x = ((x - MEAN) / STD).unsqueeze(0).unsqueeze(0).to(device)   # (1, 1, 28, 28)

    t0 = time.perf_counter()
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].cpu()           # logit -> olasılık
    ms = (time.perf_counter() - t0) * 1000

    scores = {DISPLAY_NAMES[i]: float(probs[i]) for i in range(len(DISPLAY_NAMES))}
    preview = Image.fromarray(arr28).resize((224, 224), Image.NEAREST)   # büyütülmüş önizleme
    return scores, preview, f"{ms:.1f} ms (yalnızca model, ön işleme hariç)"


demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="pil", label="Kıyafet fotoğrafı"),
        gr.Radio(choices=BG_MODES, value="Otomatik", label="Arka plan"),
    ],
    outputs=[
        gr.Label(num_top_classes=3, label="Tahmin"),
        gr.Image(label="Modelin gördüğü görüntü (28x28)"),
        gr.Textbox(label="Tahmin süresi"),
    ],
    title="Fashion-MNIST Sınıflandırıcı (CNN)",
    description=(
        "Fashion-MNIST ile eğitilmiş basit bir CNN. Kıyafeti sade bir yüzeye koyup ortalayarak fotoğrafla "
        "ve yükle. 10 sınıf: tişört, pantolon, kazak, elbise, mont, sandalet, gömlek, spor ayakkabı, çanta, bilekli bot."
    ),
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5892cb55ccab06bf86.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5892cb55ccab06bf86.gradio.live


## Notlar

- `share=True` geçici bir genel link üretir; Colab oturumu açık olduğu sürece çalışır. Uygulamayı kapatmak için hücreyi durdurunuz.
- **Arka plan** seçeneği: *Otomatik* çoğu zaman yeterli. Önizlemede nesne koyu, zemin açık görünüyorsa elle **"Açık arka plan (ters çevir)"** seçiniz.